# HNL mu-pi: BNB MC + off-beam + data + HNL (cafpybara)

Same frames, cuts (`ana.DEFAULT_CUTS`) and variables as `hnl_mupi_selection.ipynb`, with the Run 3 fixed-dev data added
(structure follows the nueCC `signal_plots` and hnlpi0 `hnl_plotting` notebooks):

* Prediction = BNB nu + CORSIKA cosmics MC (weight 1) + off-beam+light data (weight `weights_mc` = the off-beam scale to the BNB MC POT),
  all times `POT_DATA / POT_BNB`. The off-beam scale is `(1 - f) * (on-beam gates per POT * POT_BNB) / off-beam gates`, `f = config.OFFBEAM_F`,
  as in nueCC and hnlpi0.
* Data = on-beam BNB+light (unblinded Run 3 fixed-dev subset, ~5e18 POT), Poisson errors. The Data/Pred band is the weighted stack's statistical
  error (BNB MC + off-beam). No systematics yet.
* HNL = step line at the simulated |U_mu4|^2 (MeVPrtl weights) scaled to the data POT, times `ana.HNL_DISPLAY_SCALE[m]`. One figure per mass.
* Data timing: `preprocess_databnb` (per-run-period calibration, drops slices without RWM timing and bad periods), `preprocess_dataoff` (off-beam calibration).
* PDFs go to `PLOT_DIR` as `<level>_<var>_m<mass>.pdf`.

In [ ]:
import sys
from pathlib import Path

sys.path[:0] = ["/exp/sbnd/app/users/sdas1/cafpyana", "/exp/sbnd/app/users/sdas1"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import cafpybara.analyses.hnl_mupi as ana
from cafpybara.analyses.hnl_mupi import config

PLOT_DIR = Path("/exp/sbnd/app/users/sdas1/hnl/plots/selection_study_cafpybara_data")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
MASSES = (300, 365)
CUTS = ana.DEFAULT_CUTS     # same cuts as hnl_mupi_selection.ipynb
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
c = lambda *p: tuple(list(p) + [""] * (6 - len(p)))   # 6-level column key


## 1. Load (cafpyana preselection already applied)

In [ ]:
STREAM = True   # read the frames from dCache over xrootd (no /pnfs mount, e.g. on JupyterHub)
if STREAM:
    ana.ensure_token()
    url = ana.xrootd_url
else:
    url = lambda p: p
bnb0, POT_BNB, _ = ana.load_mc(url(config.BNB_FILE))
hnl0, POT_HNL, HNL_INFO = {}, {}, {}
for m in MASSES:
    hnl0[m], POT_HNL[m], HNL_INFO[m] = ana.load_mchnl(url(config.HNL_FILES[m]))
data0, POT_DATA, NGATES_ON = ana.load_data(url(config.DATA_BNB_FILE), onbeam=True)
off0, _, NGATES_OFF = ana.load_data(url(config.DATA_OFFBEAM_FILE), onbeam=False)

W = POT_DATA / POT_BNB                                   # BNB MC -> data POT
W_OFF = (1 - config.OFFBEAM_F) * (NGATES_ON / POT_DATA * POT_BNB) / NGATES_OFF   # off-beam slice -> BNB MC POT
off0[("weights_mc", "", "", "", "", "")] = W_OFF
W_HNL = {m: POT_DATA / POT_HNL[m] for m in MASSES}        # times weights_mc
print(f"Data: {len(data0)} slices, POT = {POT_DATA:.3e}, on-beam gates = {NGATES_ON:.4g}; off-beam: {len(off0)} slices, gates = {NGATES_OFF:.4g}")
print(f"BNB: {len(bnb0)} slices, POT = {POT_BNB:.3e}; scale to data = {W:.4f}; off-beam weight = {W_OFF:.4f} (x {W:.4f})")
for m, h in hnl0.items():
    sig = h.signal == ana.signal_dict['hnl']
    print(f"m{m}: {len(h)} slices, {int(sig.sum())} HNL-matched, |U|^2 = {HNL_INFO[m]['simU']:.1e}, "
          f"expected matched slices @ {POT_DATA:.2e} POT = {h.weights_mc[sig].to_numpy().sum() * W_HNL[m]:.3g}")

## 2. Cut flow
Slices per sample after each cut, scaled to the data POT; off-beam and the prediction include their scale factors.

In [ ]:
sel = lambda df: ana.select(df, cuts=CUTS, savedict=True, check_preprocessed=False)
st = {"bnb": {"preselection": bnb0, **sel(bnb0)}, "off": {"preselection": off0, **sel(off0)},
      "data": {"preselection": data0, **sel(data0)}}
rows = []
for stage, label in {"preselection": "cafpyana preselection", **{c.name: c.label for c in CUTS}}.items():
    b, o, d = len(st["bnb"][stage]) * W, len(st["off"][stage]) * W_OFF * W, len(st["data"][stage])
    rows.append({"cut": label, "BNB MC": b, "off-beam": o, "prediction": b + o, "data": d, "data/pred": d / (b + o)})
display(pd.DataFrame(rows).set_index("cut").round(3))
display(ana.cutflow({"bnb": bnb0, **{f"m{m}": h for m, h in hnl0.items()}}, CUTS,
                    weights={"bnb": W, **{f"m{m}": W_HNL[m] for m in MASSES}}).round(3))
bnb, off, data = (ana.select(d, CUTS) for d in (bnb0, off0, data0))
pred = pd.concat([bnb, off], ignore_index=True)       # BNB MC (weights_mc = 1) + off-beam (weights_mc = W_OFF)
hnl = {m: ana.select(h, CUTS) for m, h in hnl0.items()}
hnl_sig = {m: h[h.signal == ana.signal_dict["hnl"]] for m, h in hnl.items()}

## 3. Plot helper

In [ ]:
def plot(name, col, xlabel, bins=None, logy=False):
    """One PDF per mass. bins=None: 50 bins over the 0.5-99.5 percentiles of the prediction, data and HNL."""
    if bins is None:
        bins = ana.auto_bins([pred, data, *hnl_sig.values()], col)
        if bins is None:
            print("no valid entries:", name); return
    for m in MASSES:
        k = ana.HNL_DISPLAY_SCALE[m]
        res = ana.plot_mc_hnl_data(pred, hnl_sig[m], data, col, bins, scale_nu=W, scale_hnl=W_HNL[m] * k,
                                   hnl_label=ana.hnl_label(m, HNL_INFO[m]["simU"], k), log_y=logy,
                                   overflow=False, xlabel=xlabel, title=" ", counts=True,
                                   ylabel=f"Slices / bin ({POT_DATA:.2e} POT)",
                                   legend_kwargs={"fontsize": 8},
                                   savefig=PLOT_DIR / f"{name}_m{m}.pdf")
        plt.show()
        plt.close(res[0])

## 4. Slice-level variables (after the cuts)

In [ ]:
P = 18.936  # bunch period [ns]
SLICE_VARS = [
    ("nu_score",            c("slc", "nu_score"),                       "Pandora ν score",           np.linspace(0.5, 1, 51),  True),
    ("bcfm_score",          c("slc", "barycenterFM", "score"),          "BCFM score",                np.linspace(0, 2, 81),    True),
    ("bcfm_score_zoom",     c("slc", "barycenterFM", "score"),          "BCFM score (zoom)",         np.linspace(0, 0.3, 61),  True),
    ("bcfm_deltaZ",         c("slc", "barycenterFM", "deltaZ"),         "BCFM ΔZ [cm]",              None,                     True),
    ("bcfm_chi2",           c("slc", "barycenterFM", "chi2"),           "BCFM χ²",                   None,                     True),
    ("bcfm_chargeTotal",    c("slc", "barycenterFM", "chargeTotal"),    "BCFM total charge",         None,                     True),
    ("bcfm_flashPEs",       c("slc", "barycenterFM", "flashPEs"),       "BCFM flash PEs",            None,                     True),
    ("bcfm_flashTime_corr", c("slc", "barycenterFM", "flashTime"),      "BCFM flash time, generator-corrected [µs]", None, False),
    ("bcfm_flashTime_phase", c("slc", "barycenterFM", "flashTime_calib_mod"), "(flash time - vertex ToF) mod 18.936 ns [ns]",
     np.linspace(0, P, 25), False),
    ("opt0_score",          c("slc", "opt0", "score"),                  "opt0 score",                None,                     True),
    ("opt0_time",           c("slc", "opt0", "time"),                   "opt0 time [µs]",            None,                     False),
    ("opt0_measPE",         c("slc", "opt0", "measPE"),                 "opt0 measured PE",          None,                     True),
    ("opt0_hypoPE",         c("slc", "opt0", "hypoPE"),                 "opt0 hypothesis PE",        None,                     True),
    ("crlongtrkdiry",       c("slc", "nuid", "crlongtrkdiry"),          "CR long-track dir. y",      None,                     False),
    ("vtx_x",               c("slc", "vertex", "x"),                    "Slice vertex x [cm]",       np.linspace(-200, 200, 41), False),
    ("vtx_y",               c("slc", "vertex", "y"),                    "Slice vertex y [cm]",       np.linspace(-200, 200, 41), False),
    ("vtx_z",               c("slc", "vertex", "z"),                    "Slice vertex z [cm]",       np.linspace(0, 500, 51),  False),
    ("n_trk",               c("n_trk"),                                 "Track-like PFPs per slice", np.arange(-0.5, 12.5, 1), False),
]
for name, col, xlabel, bins, logy in SLICE_VARS:
    plot("slc_" + name, col, xlabel, bins, logy)

## 5. Longest and second-longest track per slice
Ranked by length only; not a muon/pion assignment.

In [ ]:
TRACK_VARS = [
    ("len",                 ("trk", "len"),                       "Track length [cm]",             np.linspace(0, 400, 81), True),
    ("trackScore",          ("trackScore",),                      "Track score",                   np.linspace(0.5, 1, 51), True),
    ("theta_z",             ("theta_z",),                         "Angle to z [deg]",              np.linspace(0, 180, 37), False),
    ("dist_vtx",            ("dist_vtx",),                        "Start-to-slice-vertex [cm]",    np.linspace(0, 100, 51), True),
    ("rangeP_mu",           ("trk", "rangeP", "p_muon"),          "rangeP (μ hypothesis) [GeV/c]", np.linspace(0, 2, 51),   True),
    ("rangeP_pi",           ("trk", "rangeP", "p_pion"),          "rangeP (π hypothesis) [GeV/c]", np.linspace(0, 2, 51),   True),
    ("chi2_muon_pl2",       ("trk", "chi2pid", "I2", "chi2_muon"),   "χ²(μ), plane 2",   None, True),
    ("chi2_pion_pl2",       ("trk", "chi2pid", "I2", "chi2_pion"),   "χ²(π), plane 2",   None, True),
    ("chi2_proton_pl2",     ("trk", "chi2pid", "I2", "chi2_proton"), "χ²(p), plane 2",   None, True),
    ("pida_pl2",            ("trk", "chi2pid", "I2", "pida"),        "PIDA, plane 2",    None, True),
    ("chi2_mu_over_pi_pl2", ("chi2_mu_over_pi_pl2",),             "χ²(μ)/χ²(π), plane 2",          np.linspace(0, 5, 51),   True),
]
for level, tag in [("long1", "Longest"), ("long2", "2nd-longest")]:
    for name, sub, xlabel, bins, logy in TRACK_VARS:
        plot(f"{level}_{name}", c(level, *sub), f"{tag} track: {xlabel}", bins, logy)